---

In [1]:
# gene level

In [2]:
library(AnnotationDbi)
library(org.Hs.eg.db)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(dplyr)
library(rtracklayer)

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Biobase")', and for packages 'citation("pkgname")'.


Loading required package: IRanges

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The f

##### first up regulated

In [1]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/upregulated_genes_deseq_auxincpi.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [3]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:many mapping between keys and columns



In [4]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
gene_bodies <- genes(txdb)   # direct gene coordinates, keyed by Entrez IDs
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [5]:
library(GenomicFeatures)

txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

# 1) One range per gene (Entrez IDs); ambiguous genes dropped by default
gene_bodies <- genes(txdb)                       # GRanges
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

# 2) Convert to BED6 (0-based start)
tss_df <- as.data.frame(gene_bodies)
bed_gene_df <- data.frame(
  chrom      = tss_df$seqnames,
  chromStart = tss_df$start - 1L,   # BED is 0-based, half-open
  chromEnd   = tss_df$end,
  name       = tss_df$gene_id,      # <- correct column name
  score      = 0,
  strand     = as.character(tss_df$strand),
  stringsAsFactors = FALSE
)

stopifnot(nrow(bed_gene_df) == nrow(tss_df))     # sanity check

write.table(
  bed_gene_df,
  file = "upregulated_genelevel_genebodies_auxincpi.bed",
  sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE
)

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [6]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<int>,<int>,<chr>,<dbl>,<chr>
chr19,58345177,58362751,1,0,-
chr11,65044817,65058553,10004,0,-
chr9,21994138,22128103,100048912,0,+
chr16,68644992,68727468,1001,0,+
chr20,63953383,63956985,100113386,0,+
chrX,147909430,147911817,100126270,0,-
chr5,137647571,137647649,100126343,0,-
chr8,90958470,90985238,100127983,0,-
chr10,94278680,94287478,100128054,0,-


##### now downregulated

In [7]:
mygenes <- read.delim("/usr/users/papantonis1/aman/rnaseq_data/downregulated_genes_deseq_auxincpi.txt", header = TRUE)
mygenes <- mygenes$hgnc_symbol

In [8]:
library(org.Hs.eg.db)

# Map gene symbols to Entrez IDs
mapped <- mapIds(org.Hs.eg.db,
                 keys = mygenes,
                 column = "ENTREZID",
                 keytype = "SYMBOL",
                 multiVals = "first")

# Remove NAs
entrez_ids <- na.omit(mapped)

'select()' returned 1:1 mapping between keys and columns



In [9]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene

gene_bodies <- genes(txdb)   # direct gene coordinates, keyed by Entrez IDs
gene_bodies <- gene_bodies[names(gene_bodies) %in% entrez_ids]

  2135 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [ ]:
#as.data.frame(gene_bodies)

In [10]:
as.data.frame(gene_bodies)

,seqnames,start,end,width,strand,gene_id
,<fct>,<int>,<int>,<int>,<fct>,<chr>
100009676,chr3,101676424,101679217,2794,+,100009676
10001,chr14,70581257,70641204,59948,-,10001
100049716,chr12,630858,664196,33339,+,100049716
10008,chr11,74454841,74467729,12889,-,10008
10009,chrX,120250752,120258398,7647,+,10009
100093630,chr4,118278703,118285316,6614,+,100093630
10010,chr2,161136908,161236230,99323,+,10010
100101490,chr20,31547379,31548081,703,+,100101490
100113389,chr14,45110883,45110968,86,+,100113389


In [11]:
# gene_bodies is a GRanges from: gene_bodies <- genes(txdb)
tss_df <- as.data.frame(gene_bodies)   # cols: seqnames, start, end, strand, gene_id, ...

# Build BED6 (0-based, half-open)
bed_gene_df <- data.frame(
  chrom      = tss_df$seqnames,
  chromStart = as.integer(tss_df$start) - 1L,   # BED is 0-based
  chromEnd   = as.integer(tss_df$end),
  name       = as.character(tss_df$gene_id),    # <-- FIXED
  score      = 0,
  strand     = as.character(tss_df$strand),
  stringsAsFactors = FALSE
)

write.table(
  bed_gene_df,
  file = "downregulated_gene_level_genebodies_auxincpi.bed",
  sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE
)


In [12]:
bed_gene_df

chrom,chromStart,chromEnd,name,score,strand
<fct>,<int>,<int>,<chr>,<dbl>,<chr>
chr3,101676423,101679217,100009676,0,+
chr14,70581256,70641204,10001,0,-
chr12,630857,664196,100049716,0,+
chr11,74454840,74467729,10008,0,-
chrX,120250751,120258398,10009,0,+
chr4,118278702,118285316,100093630,0,+
chr2,161136907,161236230,10010,0,+
chr20,31547378,31548081,100101490,0,+
chr14,45110882,45110968,100113389,0,+


In [ ]:
# gene-id stored as column "name"